# S8 - Actividad autónoma 4.1: streaming sobre el Proyecto Sello (equipo LLSW3)

**Proyecto Sello:** Sistema de monitoreo Big Data del tráfico de red del campus universitario (UPeU Juliaca).

**Fuente real:** topic `red-telemetria` (S7). El agente `uso-red/agente/agente_red.py` lee cada segundo los contadores
de las dos tarjetas de red del nodo de captura —Ethernet Realtek PCIe GbE (`llsw3-ethernet-2`, enlace cableado de 1 Gbps)
y Wi-Fi Intel Wi-Fi 6E AX211 (`llsw3-wi-fi-2`)— y los publica por MQTT; el puente los reenvía a Kafka.
Ninguna lectura de este notebook es simulada.

## 1. SparkSession con el conector de Kafka

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("s08-proyecto-red-telemetria")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0")
    .config("spark.sql.shuffle.partitions", "3")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
# FATAL: al detener una consulta a mitad de micro-lote Spark registra TaskKilledException (ruido esperado, ver guía 3.5)
spark.sparkContext.setLogLevel("FATAL")

KAFKA = "kafka:9092"
TOPIC = "red-telemetria"
BASE = "/opt/s08-streaming-estructurado/artifacts/red"
spark.version

:: loading settings :: url = jar:file:/usr/local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-68dd0b4e-722c-445c-b784-349c79983205;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 in central


	found org.apache.kafka#kafka-clients;3.9.2 in central
	found at.yawk.lz4#lz4-java;1.11.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.5.0 in central
	found org.apache.hadoop#hadoop-client-api;3.5.0 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.modules#scala-parallel-collections_2.13;1.2.0 in central
	found org.apache.commons#commons-pool2;2.13.1 in central
:: resolution report :: resolve 296ms :: artifacts dl 8ms
	:: modules in use:
	at.yawk.lz4#lz4-java;1.11.0 from central in [default]
	com.google.code.findbugs#jsr305;3.0.0 from central in [default]
	org.apache.commons#commons-pool2;2.13.1 from central in [default]
	org.apache.hadoop#hadoop-client-api;3.5.0 from central in [default]
	org.apache.hadoop#hadoop-client-runtime;3.5.0 from central in [default]
	org.apache.kafka#kafka-clients;3.9.2 from central in [default]
	org.apac

26/10/03 14:56:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


'4.2.0'

## 2. Verificar el topic con una lectura batch

In [2]:
lote = (
    spark.read.format("kafka")
    .option("kafka.bootstrap.servers", KAFKA)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "earliest")
    .load()
)
print("mensajes en red-telemetria:", lote.count())
(lote.selectExpr("CAST(key AS STRING) AS key", "CAST(value AS STRING) AS value", "partition", "offset")
     .where("key IN ('llsw3-ethernet-2', 'llsw3-wi-fi-2')")
     .orderBy("offset", ascending=False)
     .show(4, truncate=110))

mensajes en red-telemetria: 6438


+----------------+--------------------------------------------------------------------------------------------------------------+---------+------+
|             key|                                                                                                         value|partition|offset|
+----------------+--------------------------------------------------------------------------------------------------------------+---------+------+
|llsw3-ethernet-2|{"tipoEvento": "red.lectura", "sensorId": "llsw3-ethernet-2", "interfaz": "Ethernet 2", "tipoInterfaz": "fi...|        1|  3696|
|llsw3-ethernet-2|{"tipoEvento": "red.lectura", "sensorId": "llsw3-ethernet-2", "interfaz": "Ethernet 2", "tipoInterfaz": "fi...|        1|  3695|
|llsw3-ethernet-2|{"tipoEvento": "red.lectura", "sensorId": "llsw3-ethernet-2", "interfaz": "Ethernet 2", "tipoInterfaz": "fi...|        1|  3694|
|llsw3-ethernet-2|{"tipoEvento": "red.lectura", "sensorId": "llsw3-ethernet-2", "interfaz": "Ethernet 2", "tipoInterfa

## 3. Lectura streaming con esquema explícito (contrato `red.lectura`)

El esquema es el mismo contrato que el consumer de S7 valida. `ts` (tiempo de **evento**) sale del campo `timestamp`
que pone el agente al leer la tarjeta, no del momento en que Spark procesa el mensaje.

In [3]:
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StringType, DoubleType, LongType

esquema_red = (
    StructType()
    .add("tipoEvento", StringType())
    .add("sensorId", StringType())
    .add("interfaz", StringType())
    .add("tipoInterfaz", StringType())
    .add("velocidadEnlaceMbps", LongType())
    .add("bytesRecibidosSeg", DoubleType())
    .add("bytesEnviadosSeg", DoubleType())
    .add("paquetesRecibidosSeg", DoubleType())
    .add("paquetesEnviadosSeg", DoubleType())
    .add("erroresEntrada", LongType())
    .add("erroresSalida", LongType())
    .add("descartesEntrada", LongType())
    .add("descartesSalida", LongType())
    .add("intervaloMs", LongType())
    .add("origen", StringType())
    .add("timestamp", LongType())
)

crudo = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", KAFKA)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "latest")
    .load()
)

lecturas = (
    crudo.selectExpr("CAST(value AS STRING) AS json_str")
    .select(from_json(col("json_str"), esquema_red).alias("e"))
    .select("e.*")
    .where(col("tipoEvento") == "red.lectura")
    .withColumn("ts", (col("timestamp") / 1000).cast("timestamp"))
)
print("es un stream:", lecturas.isStreaming)
lecturas.printSchema()

es un stream: True
root
 |-- tipoEvento: string (nullable = true)
 |-- sensorId: string (nullable = true)
 |-- interfaz: string (nullable = true)
 |-- tipoInterfaz: string (nullable = true)
 |-- velocidadEnlaceMbps: long (nullable = true)
 |-- bytesRecibidosSeg: double (nullable = true)
 |-- bytesEnviadosSeg: double (nullable = true)
 |-- paquetesRecibidosSeg: double (nullable = true)
 |-- paquetesEnviadosSeg: double (nullable = true)
 |-- erroresEntrada: long (nullable = true)
 |-- erroresSalida: long (nullable = true)
 |-- descartesEntrada: long (nullable = true)
 |-- descartesSalida: long (nullable = true)
 |-- intervaloMs: long (nullable = true)
 |-- origen: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- ts: timestamp (nullable = true)



## 4. Agregación por ventana fija de 10 s con watermark de 10 s

**Métrica:** volumen de tráfico de la interfaz (bytes/s recibidos y enviados, promedio y pico por ventana) — la
dimensión "volumen de tráfico" del Proyecto Sello.

**Por qué 10 s de watermark:** el agente publica una lectura por segundo y el trayecto agente → MQTT → puente → Kafka
medido en S7 tarda milisegundos, pero en S7 también se midió una reconexión real del Wi-Fi de ~6 s
(`intervaloMs` = 6 183). Un margen de 10 s cubre ese corte sin perder lecturas; uno mayor solo retendría más estado
sin beneficio, porque una lectura con más de 10 s de retraso ya no sirve para un panel de carga en vivo.

Mientras corre la consulta, el nodo descarga un archivo grande por HTTP (tráfico real) para que la carga se vea en la tarjeta por la que sale el tráfico.

In [4]:
import threading, time, urllib.request
from pyspark.sql.functions import window, avg, max as fmax, count, round as fround

def descarga_real(segundos):
    fin = time.time() + segundos
    try:
        with urllib.request.urlopen("https://proof.ovh.net/files/1Gb.dat", timeout=10) as r:
            while time.time() < fin and r.read(1 << 16):
                pass
    except Exception as ex:
        print("descarga:", ex)

ventanas_red = (
    lecturas
    .withWatermark("ts", "10 seconds")
    .groupBy(window(col("ts"), "10 seconds"), col("sensorId"))
    .agg(
        fround(avg("bytesRecibidosSeg"), 1).alias("rx_prom_Bps"),
        fround(fmax("bytesRecibidosSeg"), 1).alias("rx_pico_Bps"),
        fround(avg("bytesEnviadosSeg"), 1).alias("tx_prom_Bps"),
        count("*").alias("n"),
    )
)

threading.Thread(target=descarga_real, args=(12,), daemon=True).start()
consulta = (
    ventanas_red.writeStream
    .outputMode("update")
    .format("console")
    .option("truncate", "false")
    .option("checkpointLocation", f"{BASE}/chk_ventanas")
    .trigger(processingTime="5 seconds")
    .start()
)
consulta.awaitTermination(40)
consulta.stop()

-------------------------------------------
Batch: 0
-------------------------------------------
+------+--------+-----------+-----------+-----------+---+
|window|sensorId|rx_prom_Bps|rx_pico_Bps|tx_prom_Bps|n  |
+------+--------+-----------+-----------+-----------+---+
+------+--------+-----------+-----------+-----------+---+



-------------------------------------------
Batch: 1
-------------------------------------------
+------------------------------------------+-------------+-----------+-----------+-----------+---+
|window                                    |sensorId     |rx_prom_Bps|rx_pico_Bps|tx_prom_Bps|n  |
+------------------------------------------+-------------+-----------+-----------+-----------+---+
|{2026-10-03 14:57:00, 2026-10-03 14:57:10}|llsw3-wi-fi-2|520732.0   |520732.0   |7095.6     |1  |
|{2026-10-03 14:56:50, 2026-10-03 14:57:00}|llsw3-wi-fi-2|20722.8    |20722.8    |2668.9     |1  |
+------------------------------------------+-------------+-----------+-----------+-----------+---+



-------------------------------------------
Batch: 2
-------------------------------------------
+------------------------------------------+-------------+-----------+------------+-----------+---+
|window                                    |sensorId     |rx_prom_Bps|rx_pico_Bps |tx_prom_Bps|n  |
+------------------------------------------+-------------+-----------+------------+-----------+---+
|{2026-10-03 14:57:00, 2026-10-03 14:57:10}|llsw3-wi-fi-2|8825226.3  |1.35680207E7|23951.8    |5  |
+------------------------------------------+-------------+-----------+------------+-----------+---+



-------------------------------------------
Batch: 3
-------------------------------------------
+------------------------------------------+-------------+-----------+------------+-----------+---+
|window                                    |sensorId     |rx_prom_Bps|rx_pico_Bps |tx_prom_Bps|n  |
+------------------------------------------+-------------+-----------+------------+-----------+---+
|{2026-10-03 14:57:00, 2026-10-03 14:57:10}|llsw3-wi-fi-2|1.0915572E7|1.35680207E7|25527.6    |10 |
+------------------------------------------+-------------+-----------+------------+-----------+---+



-------------------------------------------
Batch: 4
-------------------------------------------
+------------------------------------------+-------------+-----------+-----------+-----------+---+
|window                                    |sensorId     |rx_prom_Bps|rx_pico_Bps|tx_prom_Bps|n  |
+------------------------------------------+-------------+-----------+-----------+-----------+---+
|{2026-10-03 14:57:10, 2026-10-03 14:57:20}|llsw3-wi-fi-2|2873648.9  |7536619.1  |3054.8     |5  |
+------------------------------------------+-------------+-----------+-----------+-----------+---+



-------------------------------------------
Batch: 5
-------------------------------------------
+------------------------------------------+-------------+-----------+-----------+-----------+---+
|window                                    |sensorId     |rx_prom_Bps|rx_pico_Bps|tx_prom_Bps|n  |
+------------------------------------------+-------------+-----------+-----------+-----------+---+
|{2026-10-03 14:57:10, 2026-10-03 14:57:20}|llsw3-wi-fi-2|1596743.9  |7536619.1  |1884.7     |9  |
+------------------------------------------+-------------+-----------+-----------+-----------+---+



-------------------------------------------
Batch: 6
-------------------------------------------
+------------------------------------------+-------------+-----------+-----------+-----------+---+
|window                                    |sensorId     |rx_prom_Bps|rx_pico_Bps|tx_prom_Bps|n  |
+------------------------------------------+-------------+-----------+-----------+-----------+---+
|{2026-10-03 14:57:20, 2026-10-03 14:57:30}|llsw3-wi-fi-2|330.0      |425.1      |258.6      |5  |
+------------------------------------------+-------------+-----------+-----------+-----------+---+



-------------------------------------------
Batch: 7
-------------------------------------------
+------------------------------------------+-------------+-----------+-----------+-----------+---+
|window                                    |sensorId     |rx_prom_Bps|rx_pico_Bps|tx_prom_Bps|n  |
+------------------------------------------+-------------+-----------+-----------+-----------+---+
|{2026-10-03 14:57:20, 2026-10-03 14:57:30}|llsw3-wi-fi-2|348.5      |656.5      |289.7      |10 |
+------------------------------------------+-------------+-----------+-----------+-----------+---+



-------------------------------------------
Batch: 8
-------------------------------------------
+------------------------------------------+-------------+-----------+-----------+-----------+---+
|window                                    |sensorId     |rx_prom_Bps|rx_pico_Bps|tx_prom_Bps|n  |
+------------------------------------------+-------------+-----------+-----------+-----------+---+
|{2026-10-03 14:57:30, 2026-10-03 14:57:40}|llsw3-wi-fi-2|578.5      |1167.1     |511.0      |5  |
+------------------------------------------+-------------+-----------+-----------+-----------+---+



Con watermark y `outputMode("update")`, cada micro-lote muestra solo las ventanas que **cambiaron**: las ventanas ya
cerradas no se vuelven a imprimir ni ocupan memoria. Durante la descarga, `rx_prom_Bps` de la tarjeta activa sube a millones de bytes/s.

## 5. Checkpoint: qué guardó la consulta de ventanas

In [5]:
import os
chk = f"{BASE}/chk_ventanas"
for carpeta in sorted(os.listdir(chk)):
    print(carpeta)
ultimo = sorted(os.listdir(f"{chk}/offsets"), key=lambda x: int(x) if x.isdigit() else -1)[-1]
print("\n--- offsets/" + ultimo)
with open(f"{chk}/offsets/{ultimo}") as f:
    lineas = f.read().splitlines()
print(lineas[0]); print(lineas[1][:120] + " ..."); print(lineas[2])
print("\n--- commits:", sorted(os.listdir(f"{chk}/commits"), key=lambda x: (len(x), x))[-3:])
print("--- state:", sorted(os.listdir(f"{chk}/state/0")))

.metadata.crc
commits
metadata
offsets
sources
state

--- offsets/8
v1
{"batchWatermarkMs":1791039439259,"batchTimestampMs":1791039455005,"conf":{"spark.sql.streaming.stateStore.providerClass ...
{"red-telemetria":{"0":746,"1":3697,"2":2032}}

--- commits: ['.6.crc', '.7.crc', '.8.crc']
--- state: ['0', '1', '2', '_metadata']


## 6. Dato tardío real descartado por el watermark

Se toma una lectura **real** de la tarjeta que ya estaba en el topic desde una sesión anterior y se vuelve a publicar
tal cual (con su `timestamp` original). Para la consulta es un evento que llega muy tarde. Se publica **después** de que
la consulta ya vio lecturas en vivo, para no caer en el caso de "arranque en frío" del watermark.

In [6]:
from pyspark.sql.functions import to_json, struct, lit

antigua = (
    lote.where("CAST(key AS STRING) = 'llsw3-wi-fi-2'")
    .orderBy("offset")
    .limit(1)
    .selectExpr("CAST(key AS STRING) AS key", "CAST(value AS STRING) AS value")
    .collect()[0]
)

In [7]:
import json, datetime
evento_tardio = json.loads(antigua["value"])
fecha_tardia = datetime.datetime.utcfromtimestamp(evento_tardio["timestamp"] / 1000)
print("lectura real a reenviar:", antigua["value"])
print("tiempo de evento (UTC):", fecha_tardia)

vistas = []
def guardar_lote(df, batch_id):
    filas = df.collect()
    vistas.extend(filas)
    print(f"--- micro-lote {batch_id}: {len(filas)} ventanas actualizadas")
    for f in filas:
        print(f"    {f['window']['start']} - {f['window']['end']}  {f['sensorId']}  n={f['n']}")

def publicar_tardio(espera):
    time.sleep(espera)
    (spark.createDataFrame([(antigua["key"], antigua["value"])], ["key", "value"])
        .write.format("kafka").option("kafka.bootstrap.servers", KAFKA).option("topic", TOPIC).save())
    print(">>> reenviada la lectura real con tiempo de evento", fecha_tardia)

conteo_tardio = (
    lecturas.withWatermark("ts", "10 seconds")
    .groupBy(window(col("ts"), "10 seconds"), col("sensorId"))
    .agg(count("*").alias("n"))
)
threading.Thread(target=publicar_tardio, args=(18,), daemon=True).start()
consulta = (
    conteo_tardio.writeStream.outputMode("update")
    .foreachBatch(guardar_lote)
    .option("checkpointLocation", f"{BASE}/chk_tardio")
    .trigger(processingTime="4 seconds")
    .start()
)
consulta.awaitTermination(45)
consulta.stop()

dia_tardio = fecha_tardia.date()
del_dia_tardio = [v for v in vistas if v["window"]["start"].date() == dia_tardio]
print(f"\nVentanas del {dia_tardio} en toda la salida: {len(del_dia_tardio)}  -> el watermark descartó la lectura tardía")

lectura real a reenviar: {"tipoEvento": "red.lectura", "sensorId": "llsw3-wi-fi-2", "interfaz": "Wi-Fi 2", "tipoInterfaz": "fisica", "velocidadEnlaceMbps": 144, "bytesRecibidosSeg": 6912.5, "bytesEnviadosSeg": 1856.5, "paquetesRecibidosSeg": 14.6, "paquetesEnviadosSeg": 11.7, "erroresEntrada": 0, "erroresSalida": 0, "descartesEntrada": 0, "descartesSalida": 0, "intervaloMs": 1030, "origen": "agente-red", "timestamp": 1790905102037}
tiempo de evento (UTC): 2026-10-02 01:38:22.037000


--- micro-lote 0: 0 ventanas actualizadas


--- micro-lote 1: 1 ventanas actualizadas
    2026-10-03 14:57:40 - 2026-10-03 14:57:50  llsw3-wi-fi-2  n=4


--- micro-lote 2: 1 ventanas actualizadas
    2026-10-03 14:57:40 - 2026-10-03 14:57:50  llsw3-wi-fi-2  n=8


--- micro-lote 3: 2 ventanas actualizadas
    2026-10-03 14:57:40 - 2026-10-03 14:57:50  llsw3-wi-fi-2  n=10
    2026-10-03 14:57:50 - 2026-10-03 14:58:00  llsw3-wi-fi-2  n=2


...

--- micro-lote 7: 1 ventanas actualizadas
    2026-10-03 14:58:00 - 2026-10-03 14:58:10  llsw3-wi-fi-2  n=8


--- micro-lote 8: 2 ventanas actualizadas
    2026-10-03 14:58:00 - 2026-10-03 14:58:10  llsw3-wi-fi-2  n=10
    2026-10-03 14:58:10 - 2026-10-03 14:58:20  llsw3-wi-fi-2  n=2


--- micro-lote 9: 1 ventanas actualizadas
    2026-10-03 14:58:10 - 2026-10-03 14:58:20  llsw3-wi-fi-2  n=6


--- micro-lote 10: 1 ventanas actualizadas
    2026-10-03 14:58:10 - 2026-10-03 14:58:20  llsw3-wi-fi-2  n=10


--- micro-lote 11: 1 ventanas actualizadas
    2026-10-03 14:58:20 - 2026-10-03 14:58:30  llsw3-wi-fi-2  n=4

Ventanas del 2026-10-02 en toda la salida: 0  -> el watermark descartó la lectura tardía


## 7. Duplicado real filtrado con `dropDuplicates` acotado por watermark

Se toma la lectura real **más reciente** del nodo (de la tarjeta que esté activa) y se publica **dos veces** al topic (como haría un reintento del
puente). La consulta deduplica por (`sensorId`, `timestamp`) con un watermark de 30 s.

In [8]:
dedup = lecturas.withWatermark("ts", "30 seconds").dropDuplicates(["sensorId", "timestamp"])

consulta = (
    dedup.writeStream.outputMode("append")
    .format("memory").queryName("red_dedup")
    .option("checkpointLocation", f"{BASE}/chk_dedup")
    .trigger(processingTime="3 seconds")
    .start()
)
time.sleep(8)
reciente = (
    spark.read.format("kafka").option("kafka.bootstrap.servers", KAFKA).option("subscribe", TOPIC)
    .option("startingOffsets", "earliest").load()
    .where("CAST(key AS STRING) IN ('llsw3-ethernet-2', 'llsw3-wi-fi-2')")
    .orderBy(col("timestamp").desc()).limit(1)
    .selectExpr("CAST(key AS STRING) AS key", "CAST(value AS STRING) AS value").collect()[0]
)
ts_dup = json.loads(reciente["value"])["timestamp"]
for intento in range(2):
    (spark.createDataFrame([(reciente["key"], reciente["value"])], ["key", "value"])
        .write.format("kafka").option("kafka.bootstrap.servers", KAFKA).option("topic", TOPIC).save())
print(">>> lectura real publicada DOS veces:", reciente["value"])
consulta.awaitTermination(15)
consulta.stop()

print()
print("lecturas en la salida deduplicada:", spark.sql("SELECT count(*) FROM red_dedup").collect()[0][0])
spark.sql(f"SELECT sensorId, bytesRecibidosSeg, bytesEnviadosSeg, timestamp, ts FROM red_dedup WHERE timestamp = {ts_dup}").show(truncate=False)
print(f"Apariciones de timestamp={ts_dup}:", spark.sql(f"SELECT count(*) FROM red_dedup WHERE timestamp = {ts_dup}").collect()[0][0], "(publicada 2 veces)")

>>> lectura real publicada DOS veces: {"tipoEvento": "red.lectura", "sensorId": "llsw3-wi-fi-2", "interfaz": "Wi-Fi 2", "tipoInterfaz": "fisica", "velocidadEnlaceMbps": 144, "bytesRecibidosSeg": 104.7, "bytesEnviadosSeg": 53.3, "paquetesRecibidosSeg": 1.9, "paquetesEnviadosSeg": 1.0, "erroresEntrada": 0, "erroresSalida": 0, "descartesEntrada": 0, "descartesSalida": 0, "intervaloMs": 1031, "origen": "agente-red", "timestamp": 1791039513008}



lecturas en la salida deduplicada: 22


+-------------+-----------------+----------------+-------------+-----------------------+
|sensorId     |bytesRecibidosSeg|bytesEnviadosSeg|timestamp    |ts                     |
+-------------+-----------------+----------------+-------------+-----------------------+
|llsw3-wi-fi-2|104.7            |53.3            |1791039513008|2026-10-03 14:58:33.008|
+-------------+-----------------+----------------+-------------+-----------------------+

Apariciones de timestamp=1791039513008: 1 (publicada 2 veces)


## 8. Intervalo de disparo: 1 s frente a 6 s sobre el mismo stream

In [9]:
def contar_microlotes(intervalo, segundos_totales):
    q = (
        lecturas.writeStream.outputMode("append").format("memory")
        .queryName("red_" + intervalo.replace(" ", "_"))
        .trigger(processingTime=intervalo)
        .start()
    )
    q.awaitTermination(segundos_totales)
    q.stop()
    filas = spark.sql(f"SELECT count(*) AS n FROM red_{intervalo.replace(' ', '_')}").collect()[0]["n"]
    return (q.lastProgress["batchId"] if q.lastProgress else None), filas

for intervalo in ["1 second", "6 seconds"]:
    batch_id, filas = contar_microlotes(intervalo, 15)
    print(f"Disparo cada {intervalo:>9}: último batchId en 15 s = {batch_id}  (lecturas procesadas: {filas})")

Disparo cada  1 second: último batchId en 15 s = 13  (lecturas procesadas: 14)


Disparo cada 6 seconds: último batchId en 15 s = 3  (lecturas procesadas: 13)


## 9. Hallazgo: una agregación en modo `update` no se puede escribir a Parquet

Antes de persistir se intentó volcar la agregación por ventanas directo a Parquet en modo `update`. Spark la rechaza al
arrancar: un *sink* de archivo solo admite `append`.

In [10]:
try:
    ventanas_red.writeStream.outputMode("update").format("parquet") \
        .option("path", f"{BASE}/no_valido").option("checkpointLocation", f"{BASE}/chk_no_valido").start()
except Exception as ex:
    print(type(ex).__name__ + ":", str(ex).splitlines()[0][:300])

AnalysisException: [STREAMING_OUTPUT_MODE.UNSUPPORTED_DATASOURCE] Invalid streaming output mode: update. This output mode is not supported in Data Source parquet. SQLSTATE: 42KDE


## 10. Persistir las lecturas en Parquet con checkpoint (dos corridas: acumula)

In [11]:
PARQUET = f"{BASE}/lecturas_parquet"
CHK_PARQUET = f"{BASE}/chk_parquet"

def corrida_parquet(segundos):
    q = (
        lecturas.writeStream.outputMode("append").format("parquet")
        .option("path", PARQUET).option("checkpointLocation", CHK_PARQUET)
        .trigger(processingTime="5 seconds").start()
    )
    q.awaitTermination(segundos)
    q.stop()
    return spark.read.parquet(PARQUET).count()

print("filas en Parquet tras la corrida 1:", corrida_parquet(20))
print("filas en Parquet tras la corrida 2:", corrida_parquet(20))
spark.read.parquet(PARQUET).select("sensorId", "bytesRecibidosSeg", "bytesEnviadosSeg", "ts").orderBy(col("ts").desc()).show(5, truncate=False)

filas en Parquet tras la corrida 1: 15


filas en Parquet tras la corrida 2: 39


+-------------+-----------------+----------------+-----------------------+
|sensorId     |bytesRecibidosSeg|bytesEnviadosSeg|ts                     |
+-------------+-----------------+----------------+-----------------------+
|llsw3-wi-fi-2|4447.9           |10879.5         |2026-10-03 14:59:59.338|
|llsw3-wi-fi-2|274.9            |158.3           |2026-10-03 14:59:58.304|
|llsw3-wi-fi-2|472.8            |381.3           |2026-10-03 14:59:57.276|
|llsw3-wi-fi-2|158.2            |193.4           |2026-10-03 14:59:56.249|
|llsw3-wi-fi-2|104.9            |53.4            |2026-10-03 14:59:55.224|
+-------------+-----------------+----------------+-----------------------+
only showing top 5 rows
